In [1]:
import Pkg
Pkg.add("JuMP")
Pkg.add("HiGHS")

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
Precompiling packages...
              ✗ CPLEX
  0 dependencies successfully precompiled in 3 seconds. 59 already precompiled.

The following 1 direct dependency failed to precompile:

CPLEX 

Failed to precompile CPLEX [a076750e-1247-5638-91d2-ce28b192dca0] to "/home/husted42/.julia/compiled/v1.12/CPLEX/jl_mdXUkM".
ERROR: LoadError: CPLEX not properly installed. Please run Pkg.build("CPLEX")
Stacktrace:
  [1] error(s::String)
    @ Base ./error.jl:44
  [2] top-level scope
    @ ~/.julia/packages/CPLEX/5jmjD/src/CPLEX.jl:12
  [3] include(mod::Module, _path::String)
    @ Base ./Base.jl:306
  [4] include_package_for_output(pkg::Base.PkgId, input::String, depot_path::Vector{String}, dl_load_path::Vector{String}, load_path::Vector{String}, concrete_deps::Vector{Pair{Base.P

# Micro brewery

In [2]:
using JuMP, HiGHS

########## ---------- Variables ---------- ##########
M = [
    35  15   5;
    20  10  20;
    15  20  20;
    45  15  35;
    25  15  35;
    65  55  80;
    40  90  60;
    50  80  30;
    35  25  35;
    85  45  20;
    50   5  20;
    55  30  40
]

noMonths, noProd = size(M)
storage_cost = 0.1
initStorage = [25 65 75]

########## ---------- Models ---------- ##########
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

########## ---------- Variables ---------- ##########
@variable(model, production[1:noMonths, 1:noProd] >= 0, Int) # production amout
@variable(model, beer[1:noMonths, 1:noProd], Bin) # production decision

@variable(model, storage[1:noMonths, 1:noProd] >= 0, Int) # storage units
@variable(model, sold[1:noMonths, 1:noProd] >= 0, Int) # sold units

########## ---------- Objectives ---------- ##########
@objective(model, Min, sum(storage_cost * storage[i, j] for i in 1:noMonths, j in 1:noProd))

########## ---------- Constraints ---------- ##########
# Production capacity constraints 
# We can brew 120 units of beer each month
# hold the constration: 
    # If beer[i,j] == 0 => production[i,j] == 0
    # By multiplying beer[i,j] on the right side we enforce this condition
    # + The second constraint
@constraint(model, [i in 1:noMonths, j in 1:noProd], 
    production[i, j] <= 120 * beer[i,j])

@constraint(model, [i in 1:noMonths], 
    sum(beer[i, j] for j in 1:noProd) <= 1)


# Storage balance constraints
for i in 1:noMonths
    @constraint(model, 
        [j in 1:noProd], 
        storage[i, j] == (i == 1 ? initStorage[j] : storage[i-1, j]) + production[i, j] - sold[i, j]
    )
end

# Demand constraints
for i in 1:noMonths
    @constraint(model, 
        [j in 1:noProd], 
        sold[i, j] == M[i, j]
    )
end


optimize!(model)
println("Optimal solution:")
println(objective_value(model))

println("Production plan:", Matrix(value.(production)))
println("Beer plan:", value.(beer))
println("Storage plan:", value.(storage))
println("Sales plan:", value.(sold))


Optimal solution:
192.5
Production plan:[65.0 0.0 0.0; 0.0 70.0 0.0; -0.0 0.0 90.0; 120.0 0.0 -0.0; 0.0 120.0 0.0; -0.0 -0.0 120.0; 120.0 0.0 -0.0; -0.0 120.0 -0.0; -0.0 -0.0 115.0; 85.0 -0.0 0.0; 105.0 0.0 -0.0; -0.0 30.0 0.0]
Beer plan:[1.0 0.0 0.0; -0.0 1.0 -0.0; -0.0 -0.0 1.0; 1.0 -0.0 0.0; -0.0 1.0 -0.0; -0.0 0.0 1.0; 1.0 -0.0 -0.0; -0.0 1.0 -0.0; -0.0 0.0 1.0; 1.0 -0.0 -0.0; 1.0 0.0 -0.0; 0.0 1.0 -0.0]
Storage plan:[55.0 50.0 70.0; 35.0 110.0 50.0; 20.0 90.0 120.0; 95.0 75.0 85.0; 70.0 180.0 50.0; 5.0 125.0 90.0; 85.0 35.0 30.0; 35.0 75.0 0.0; -0.0 50.0 80.0; -0.0 5.0 60.0; 55.0 -0.0 40.0; 0.0 0.0 0.0]
Sales plan:[35.0 15.0 5.0; 20.0 10.0 20.0; 15.0 20.0 20.0; 45.0 15.0 35.0; 25.0 15.0 35.0; 65.0 55.0 80.0; 40.0 90.0 60.0; 50.0 80.0 30.0; 35.0 25.0 35.0; 85.0 45.0 20.0; 50.0 5.0 20.0; 55.0 30.0 40.0]


# Santa's workshop tour 2019

In [ ]:
include("Data/SantasWorkshopData_1000_20.jl")
using JuMP, HiGHS

F=1000
D=20

println("Families ", FamilySize)
println("DayVisitCost ", DayVisitCost)
println("Families ", length(FamilySize))
println("DayVisitCost ", size(DayVisitCost))

########## ---------- Models ---------- ##########
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

########## ---------- variables ---------- ##########
@variable(model, x[1:F, 1:D], Bin) # assignment of family f to day d

########## ---------- Objectives ---------- ##########
@objective(model, Min, 
    sum(DayVisitCost[f, d] * x[f, d] for f in 1:F, d in 1:D)
)

########## ---------- Constraints ---------- ##########
# Each family is assigned to exactly one day
for f in 1:F
    @constraint(model, sum(x[f, d] for d in 1:D) == 1)
end

# Daily capacity constraints
for d in 1:D
    @constraint(model, 
        sum(FamilySize[f] * x[f, d] for f in 1:F) >= 125)
    @constraint(model, 
        sum(FamilySize[f] * x[f, d] for f in 1:F) <= 300)
end

optimize!(model)
println("Objective value: ", objective_value(model))

Families [4, 4, 3, 2, 4, 4, 2, 5, 4, 7, 7, 7, 6, 2, 2, 2, 3, 4, 3, 2, 6, 5, 6, 3, 6, 3, 2, 4, 5, 4, 3, 3, 3, 3, 3, 3, 6, 2, 5, 4, 6, 8, 8, 2, 2, 4, 3, 4, 4, 2, 6, 2, 2, 4, 3, 3, 5, 4, 4, 4, 2, 4, 6, 5, 3, 5, 8, 5, 7, 4, 3, 4, 3, 3, 4, 4, 4, 4, 5, 7, 2, 4, 4, 2, 6, 3, 3, 2, 4, 2, 6, 4, 2, 3, 3, 4, 6, 4, 4, 3, 6, 2, 7, 7, 2, 4, 4, 3, 5, 5, 6, 4, 3, 4, 6, 4, 4, 6, 4, 4, 4, 3, 6, 4, 5, 3, 5, 8, 6, 7, 6, 4, 2, 4, 8, 6, 3, 2, 5, 3, 6, 4, 3, 3, 4, 2, 3, 5, 4, 4, 4, 6, 2, 2, 4, 2, 5, 3, 6, 6, 2, 4, 5, 4, 6, 7, 4, 4, 4, 5, 7, 4, 7, 3, 6, 2, 6, 4, 2, 5, 6, 3, 2, 3, 6, 2, 4, 2, 3, 3, 4, 3, 3, 4, 2, 3, 6, 2, 7, 2, 2, 7, 5, 4, 5, 2, 2, 5, 4, 6, 4, 4, 3, 3, 3, 2, 7, 5, 3, 6, 2, 5, 3, 6, 4, 2, 2, 5, 7, 4, 2, 3, 3, 3, 4, 3, 5, 7, 4, 3, 4, 4, 2, 4, 4, 6, 6, 6, 5, 3, 8, 3, 4, 3, 3, 4, 4, 6, 4, 5, 5, 3, 4, 3, 2, 7, 7, 2, 4, 5, 6, 6, 6, 2, 3, 6, 4, 3, 3, 6, 6, 3, 3, 4, 6, 6, 4, 4, 2, 3, 5, 2, 3, 5, 7, 8, 3, 3, 3, 3, 4, 2, 4, 3, 6, 5, 6, 4, 4, 4, 3, 2, 2, 5, 3, 4, 5, 3, 4, 5, 5, 5, 4, 4, 3, 2, 4, 4, 3, 3, 

# Factory planning

In [ ]:
using JuMP
using HiGHS

profit = [10, 6, 8, 4, 11, 9, 3]

process_time = [
    0.50  0.70  0.00  0.00  0.30  0.20  0.50;   # Grinding
    0.10  0.20  0.00  0.30  0.00  0.60  0.00;   # Vertical drilling
    0.20  0.00  0.80  0.00  0.00  0.00  0.60;   # Horizontal drilling
    0.05  0.03  0.00  0.07  0.10  0.00  0.08;   # Boring
    0.00  0.00  0.01  0.00  0.05  0.00  0.05    # Planing
]

machines_avail = [4, 2, 3, 1, 1]  # [grinders, vert drills, horiz drills, borer, planer]

demand = [
    500 1000 300 300  800 200 100;
    600  500 200   0  400 300 150;
    300  600   0   0  500 400 100;
    200  300 400 500  200   0 100;
      0  100 500 100 1000 300   0;
    500  500 100 300 1100 500  60
]

products = 1:7
machines = 1:5
months   = 1:6

storage_cap  = 100.0
storage_cost = 0.5
storage_end  = 50.0

working_days = 24
shift_hours  = 8
shifts_per_day = 2
H = working_days * shifts_per_day * shift_hours   # 384 hours per machine per month

# Maintenance (machines down) by month:
# months: 1 Jan, 2 Feb, 3 Mar, 4 Apr, 5 May, 6 Jun
down = zeros(Int, 5, 6)
down[1, 1] = 1              # Jan: 1 grinder
down[3, 2] = 2              # Feb: 2 horizontal drills
down[4, 3] = 1              # Mar: 1 borer
down[2, 4] = 1              # Apr: 1 vertical drill
down[1, 5] = 1; down[2, 5] = 1  # May: 1 grinder and 1 vertical drill
down[5, 6] = 1; down[3, 6] = 1  # Jun: 1 planer and 1 horizontal drill

model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

########## ---------- Variables ---------- ##########
@variable(model, x[products, months] >= 0)   # production
@variable(model, s[products, months] >= 0)   # end-of-month inventory
@variable(model, y[products, months] >= 0)   # sold

########## ---------- Objective ---------- ##########
@objective(model, Max,
    sum(profit[p] * y[p, t] for p in products, t in months) -
    sum(storage_cost * s[p, t] for p in products, t in months)
)

########## ---------- Constraints ---------- ##########

# Demand limits
@constraint(model, [p in products, t in months], y[p, t] <= demand[t, p])

# Inventory balance (initial inventory = 0)
@constraint(model, [p in products], s[p, 1] == x[p, 1] - y[p, 1])
@constraint(model, [p in products, t in 2:6], s[p, t] == s[p, t-1] + x[p, t] - y[p, t])

# Storage capacity
@constraint(model, [p in products, t in months], s[p, t] <= storage_cap)

# Ending inventory requirement at end of June
@constraint(model, [p in products], s[p, 6] >= storage_end)

# Machine time capacities by month (capacity depends on maintenance)
@constraint(model, [k in machines, t in months],
    sum(process_time[k, p] * x[p, t] for p in products)
    <= (machines_avail[k] - down[k, t]) * H
)

optimize!(model)

println("Objective value: ", objective_value(model))


Objective value: 93715.17857142858
